In [1]:
import pandas as pd

df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()


,id,sender,subject,body,priority,triage_label
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond


In [2]:
def email_assistant(email_text):
    text = str(email_text).lower()

    if any(w in text for w in ["urgent", "asap", "deadline", "eod", "immediately"]):
        return "notify", "urgent", 0.9

    elif any(w in text for w in ["please", "can you", "could you", "review", "confirm"]):
        return "respond", "polite", 0.8

    elif any(w in text for w in ["thank you", "thanks"]):
        return "ignore", "polite", 0.7

    else:
        return "review", "neutral", 0.4


In [3]:
predictions = []

for _, row in df.iterrows():
    intent, tone, confidence = email_assistant(row["body"])

    predictions.append({
        "id": row["id"],
        "predicted_intent": intent,
        "predicted_tone": tone,
        "confidence": confidence
    })

pred_df = pd.DataFrame(predictions)
pred_df.head()


,id,predicted_intent,predicted_tone,confidence
0,1,review,neutral,0.4
1,2,respond,polite,0.8
2,3,review,neutral,0.4
3,4,respond,polite,0.8
4,5,respond,polite,0.8


In [4]:
def hitl_decision(confidence):
    if confidence >= 0.75:
        return "AUTO_APPROVED"
    else:
        return "HUMAN_REVIEW_REQUIRED"


In [5]:
pred_df["decision"] = pred_df["confidence"].apply(hitl_decision)

pred_df["final_action"] = pred_df.apply(
    lambda x: x["predicted_intent"] if x["decision"] == "AUTO_APPROVED" else "respond",
    axis=1
)

pred_df.head()


,id,predicted_intent,predicted_tone,confidence,decision,final_action
0,1,review,neutral,0.4,HUMAN_REVIEW_REQUIRED,respond
1,2,respond,polite,0.8,AUTO_APPROVED,respond
2,3,review,neutral,0.4,HUMAN_REVIEW_REQUIRED,respond
3,4,respond,polite,0.8,AUTO_APPROVED,respond
4,5,respond,polite,0.8,AUTO_APPROVED,respond


In [6]:
final_df = pred_df.merge(
    df[["id", "body"]],
    on="id",
    how="left"
)

final_df = final_df[[
    "id",
    "body",
    "predicted_intent",
    "predicted_tone",
    "confidence",
    "decision",
    "final_action"
]]

final_df.head()


,id,body,predicted_intent,predicted_tone,confidence,decision,final_action
0,1,Reminder: The client meeting is scheduled at 1...,review,neutral,0.4,HUMAN_REVIEW_REQUIRED,respond
1,2,Your invoice of INR 25515.09 is due on 2025-12...,respond,polite,0.8,AUTO_APPROVED,respond
2,3,Reminder: The client meeting is scheduled at 1...,review,neutral,0.4,HUMAN_REVIEW_REQUIRED,respond
3,4,"Hello team, please find the attached weekly re...",respond,polite,0.8,AUTO_APPROVED,respond
4,5,"Hello team, please find the attached weekly re...",respond,polite,0.8,AUTO_APPROVED,respond


In [7]:
final_df.to_csv(
    "../data/milestone3_output_Nikhitha.csv",
    index=False
)

print("Milestone 3 output saved successfully")


Milestone 3 output saved successfully


In [1]:
import os

os.chdir(r"C:\Users\nikhi\OneDrive\Documents\Internship Project\infosys-langgraph-email-assistant-group2")
print(os.getcwd())


C:\Users\nikhi\OneDrive\Documents\Internship Project\infosys-langgraph-email-assistant-group2


In [2]:
import pandas as pd
df=pd.read_csv("data/sample_emails_with_triage_200.csv")
df.head()

,id,sender,subject,body,priority,triage_label
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond


In [3]:
dangerous_tools = ["send_email","create_calender_invite"] 

In [4]:
def agent_decide_action(body):
    if "meeting" in body.lower():
        return "create_calender_invite"
    if "reply" in body.lower():
        return "send_email"
    return "read_calender"

In [5]:
def hitl_checkpoint(action):
    if action in dangerous_tools:
        return "wait_for_human"
    return "safe"

In [6]:
def human_decision():
    decision = input("Approve action? (yes/no): ")
    return decision.lower() == "yes"

In [7]:
def execute_tool(action):
    print(f"Executing tool: {action}")

In [8]:
def run_agent(body):
    print(f"Email received: {body}")
    action = agent_decide_action(body)
    print(f"Decided action: {action}")
    status = hitl_checkpoint(action)
    print(f"HITL checkpoint status: {status}")
    if status == "wait_for_human":
        decision = human_decision()
        print(f"Human decision: {decision}")
        if decision== "Approve":
            return execute_tool(action)
        elif decision == "Deny":
            return "Action denied by human."
        elif decision == "Edit":
            return "Action escalated for further review."
    return execute_tool(action)

In [ ]:
email = "Please schedule a meeting with the client."
result = run_agent(email)
print(result)

Email received: Please schedule a meeting with the client.
Decided action: create_calender_invite
HITL checkpoint status: wait_for_human
